# Part 1 - Probability Distributions: EM Algorithm

**Team Member:** Bior Majok Aguer Kuir

In [2]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# In Colab, get the group repo so the notebook can access data/ and images/.
if "google.colab" in sys.modules:
    repo_url = "https://github.com/Abdull-Kudus/ML--Formative--3-Group-20.git"
    repo_path = "/content/repo"

    if not os.path.exists(os.path.join(repo_path, ".git")):
        subprocess.run(
            ["git", "clone", "--branch", "Part1_Probability_Distributions_EM_Algorithm", "--single-branch", repo_url, repo_path],
            check=True,
        )

    os.chdir(os.path.join(repo_path, "notebooks"))

DATA_PATH = os.path.join("..", "data", "GaltonFamilies.csv")
IMAGES_DIR = os.path.join("..", "images")
LABELS = ("Children", "Fathers")
TOLERANCE = 1e-6
MAX_ITERATIONS = 1000

os.makedirs(IMAGES_DIR, exist_ok=True)

def save_and_show(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, filename), dpi=120)
    plt.show()

## 1. Data

Dataset: Kaggle - *Parents' Heights vs Children's Heights (Galton data)*: 934 rows (one per child), 205 families, heights in inches. We chose **Fathers and Children**.

* **Children:** all 934 values of `childHeight`.
* **Fathers:** 205 values of `father`, **one per family**. In the file a father is repeated on the row of every one of his children (up to 15 times), so we drop duplicates - one man, one height.

The labels (daughter / son / father) are kept aside and **never given to EM**. We only use them at the end to check the result.

In [2]:
def load_heights(path=DATA_PATH):
    # Return (heights, hidden_groups). hidden_groups is used only for checking, never by EM.
    df = pd.read_csv(path)
    fathers = df.drop_duplicates("family")
    heights = np.concatenate([df["childHeight"].to_numpy(float), fathers["father"].to_numpy(float)])
    child_groups = np.where(df["gender"] == "male", "son", "daughter")
    groups = np.concatenate([child_groups, np.full(len(fathers), "father")])
    return heights, groups


heights, groups = load_heights()
print("Rows in file: 934 children,", len(np.unique(pd.read_csv(DATA_PATH)['family'])), "families")
print("Heights given to EM:", len(heights), "(934 children + 205 fathers)")
print(f"Global mean = {heights.mean():.4f} in, min = {heights.min()}, max = {heights.max()}")

Rows in file: 934 children, 205 families
Heights given to EM: 1139 (934 children + 205 fathers)
Global mean = 67.2085 in, min = 56.0, max = 79.0


## 2. The model

Every height $x$ is assumed to come from one of two bell curves (Gaussian distributions):

$$p(x)=\pi_1\,\mathcal N(x\mid\mu_1,\sigma_1^2)+\pi_2\,\mathcal N(x\mid\mu_2,\sigma_2^2),\qquad
\mathcal N(x\mid\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\,e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

| Parameter | Meaning |
|---|---|
| $\mu_1,\mu_2$ | means (centre of each bell curve) |
| $\sigma_1^2,\sigma_2^2$ | variances (width of each curve) |
| $\pi_1,\pi_2$ | mixing coefficients: share of the data from each group, $\pi_1+\pi_2=1$ |

**Log-likelihood** - how well the current parameters explain all $n=1139$ heights (higher is better):
$$\log L=\sum_{i=1}^{n}\log\big(\pi_1\mathcal N(x_i\mid\mu_1,\sigma_1^2)+\pi_2\mathcal N(x_i\mid\mu_2,\sigma_2^2)\big)$$

In [3]:
def gaussian_pdf(x, mean, variance):
    # N(x | mean, variance)
    return np.exp(-((x - mean) ** 2) / (2.0 * variance)) / np.sqrt(2.0 * np.pi * variance)


def weighted_densities(x, params):
    # Column k holds pi_k * N(x_i | mu_k, var_k); shape (n_points, 2)
    return np.column_stack([params["pi"][k] * gaussian_pdf(x, params["mu"][k], params["var"][k])
                            for k in range(2)])


def log_likelihood(x, params):
    return float(np.sum(np.log(weighted_densities(x, params).sum(axis=1))))